# Lab 04 — CNNs, residual connections and depth

**Dataset:** [ylecun/mnist](https://huggingface.co/datasets/ylecun/mnist). We compare a small fully-connected model with a CNN and then test whether a residual block changes optimization behavior.

## The math behind the experiment

A convolution applies a small kernel across local image regions. In simplified notation,

$$
y_{i,j}=\sum_{u,v}K_{u,v}x_{i+u,j+v}+b.
$$

The same kernel is reused at many image locations. That is **parameter sharing**: the model can learn an edge detector once and use it across the image.

A residual block instead learns a correction to an existing representation:

$$
y=x+F(x).
$$

The shortcut gives gradients and information a direct path through the network, which helps very deep models train.

In [ ]:
!pip -q install datasets torch torchvision pandas
from datasets import load_dataset
import numpy as np,pandas as pd,torch
from torch import nn
from torch.utils.data import TensorDataset,DataLoader
tr=load_dataset("ylecun/mnist",split="train[:4000]"); te=load_dataset("ylecun/mnist",split="test[:1000]")
def make(ds): return torch.tensor(np.array([np.array(x) for x in ds["image"]])/255.,dtype=torch.float32).unsqueeze(1),torch.tensor(ds["label"])
Xtr,ytr=make(tr); Xte,yte=make(te); print(Xtr.shape)

## Step 1 — Prediction

CNNs should need fewer parameters than a dense network for image structure because the same local filter is reused across positions. Expect competitive or better accuracy despite fewer parameters.

In [ ]:
class Dense(nn.Module):
 def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Flatten(),nn.Linear(784,64),nn.ReLU(),nn.Linear(64,10))
 def forward(self,x): return self.net(x)
class CNN(nn.Module):
 def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Conv2d(1,16,3,padding=1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(16,32,3,padding=1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(32,10))
 def forward(self,x): return self.net(x)
def fit(m,epochs=3):
 opt=torch.optim.Adam(m.parameters(),lr=1e-3); loss=nn.CrossEntropyLoss(); dl=DataLoader(TensorDataset(Xtr,ytr),64,shuffle=True)
 for _ in range(epochs):
  for xb,yb in dl: opt.zero_grad(); loss(m(xb),yb).backward(); opt.step()
 with torch.no_grad(): acc=(m(Xte).argmax(1)==yte).float().mean().item()
 return sum(p.numel() for p in m.parameters()),acc
rows=[]
for n,m in [("dense",Dense()),("cnn",CNN())]: rows.append([n,*fit(m)])
pd.DataFrame(rows,columns=["model","parameters","test_accuracy"])

## Step 2 — Receptive-field intervention

Replace the CNN's local 3×3 kernels with a 1×1 convolution while keeping the training protocol fixed. Predict that the modified model should lose the spatial inductive bias.

In [ ]:
class CNN1x1(nn.Module):
 def __init__(self): super().__init__(); self.net=nn.Sequential(nn.Conv2d(1,16,1),nn.ReLU(),nn.MaxPool2d(2),nn.Conv2d(16,32,1),nn.ReLU(),nn.AdaptiveAvgPool2d(1),nn.Flatten(),nn.Linear(32,10))
 def forward(self,x): return self.net(x)
rows.append(["cnn_1x1",*fit(CNN1x1())]); pd.DataFrame(rows,columns=["model","parameters","test_accuracy"])

## Read the graph — accuracy must be considered with model size

The plot compares parameter count and test accuracy for the dense baseline, spatial CNN, and 1×1-convolution intervention. The 1×1 model removes most local spatial mixing, while the 3×3 model can learn patterns across neighboring pixels.

Do not conclude that one architecture is universally best from one short run. Check the parameter budget, training time, random seed, and variability before making a stronger claim.

In [ ]:
import matplotlib.pyplot as plt

model_results = pd.DataFrame(rows, columns=["model", "parameters", "test_accuracy"])
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(model_results["model"], model_results["parameters"])
axes[0].set_title("Model size")
axes[0].set_ylabel("Trainable parameters")
axes[0].tick_params(axis="x", rotation=15)
axes[0].grid(axis="y", alpha=.25)

axes[1].bar(model_results["model"], model_results["test_accuracy"])
axes[1].set_title("Held-out accuracy")
axes[1].set_ylabel("Accuracy")
axes[1].set_ylim(0, 1)
axes[1].tick_params(axis="x", rotation=15)
axes[1].grid(axis="y", alpha=.25)
fig.tight_layout()
plt.show()

## Conclusion / answer key

Expected: the CNN should exploit local spatial structure more effectively than the dense baseline at comparable scale. The 1×1 intervention removes most local spatial mixing, so any drop is evidence for the convolutional inductive bias rather than merely “more layers.”

### Research extension
Compare residual vs plain blocks at equal depth and parameter count, measuring optimization curves and final accuracy.